# Create xyz and CIF files of Catalan NPs of different sizes and forms from a dataset of compounds (CIF dataset)

## Imports

In [1]:
import os
import sys
import tkinter

print(os.getcwd())
cwd0 = './styles/'
sys.path.append(cwd0)
from pathlib import Path
import numpy as np
import visualID as vID
from visualID import  fg, hl, bg

from pyNanoMatBuilder import catalanNPs as cNP
from pyNanoMatBuilder import utils as pyNMBu
from pyNanoMatBuilder import data
import importlib

from ase import io
from ase.atoms import Atoms
from ase.geometry import cellpar_to_cell
from ase.spacegroup import get_spacegroup
from ase.io import read
from ase.io import write
from ase.visualize import view

from debyecalculator import DebyeCalculator
import pyNanoMatBuilder.utilsDC as pyNMBuDC

/home/sara/pyNanoMatBuilder


##  Class for rhombic dodecahedron (bccrDD) and dihedral rhombic dodecahedron (fccdrDD)

#### NB: the Catalan class is instantiated by defining a compound and its interatomic distance. The function FindInterAtomicDist, which is defined in utils, calculates the interatomic distance by extracting the cell paratemers and uses well known formulas for fcc, bcc and hcp. Thus, compounds with unknown interatomic distances are not used to create Platonic NPs.

<span style="color:#FF0000"> Problem : not possible to create the forms for multiple elements like TiO2 or NaCl


### Definition of the Catalan_Files class

In [2]:
class Catalan_Files :
    """
    A class for generating XYZ and CIF files of rhombic dodecahedron (bccrDD) 
    and dihedral rhombic dodecahedron (fccdrDD) Catalan nanoparticles (NPs) 
    from a dataset of compounds (CIF dataset). This process enables the creation 
    of a well-structured database optimized for machine learning applications, 
    ensuring consistency in format and data representation.
    """
    def __init__(self, path, cif_data,sizes,max_size: float=50,noOutput:bool = True):
        """
        Initialize the class with CIF data and sizes.
            Args:
        path (str) : Path that will contain the created xyz files
        cif data (DataFrame):  DataFrame containing CIF files of different compounds
        sizes (array-like): Array of the number of shells, ex: [2,3,4]
        max_size (float, optional) : Maximal size for the NPs, equals to the diameter of the circumscribed sphere, equals 50 nm by default.
        noOutput (bool): If False, details of the files are printed.
            Methods:
        create_catalan(noOutput, path): Generates Catalan nanoparticles and saves their files.
        """
        self.cif_data = cif_data  # DataFrame containing CIF
        self.loaded_cifs = {}  # Dictionary to store loaded CIF files
        self.sizes= sizes # List of sizes 

        
        self.create_catalan(noOutput,max_size,path) # Call method to generate Catalan nanoparticles
  
    def create_catalan(self,noOutput,max_size,path):
        """
        Genenrate platonic NPs and their files from cif data.
        Args:
            noOutput (bool): If False, prints details about the process.
            path (str): Path where xyz/cif files will be created.
        Note:
        - The nanoparticles are created using classes of the module 'cNP' 
        and the class 'bccrDD' and 'fccdrDD'.
        """
    
        
        for cif_name, cif_file in self.cif_data['cif file'].items():
            
            # Load cif informations
            element=cif_name.split()[0] # Extract chemical element from CIF name
            
            if not (element=='TiO2' or element=='NaCl'): # Skip TiO2 and NaCl since they are not processed
                self.cif_name=cif_name
                
                # Extract the structure name for the name of the files, for example 'Rutile' or 'Anatase' or 'Alpha'
                if len(self.cif_name.split())==2 : 
                    structure=self.cif_name.split()[1]
                else : 
                    structure=None
                
                if not noOutput :
                    print(f'\n\033[1m{cif_name.center(50)}\033[0m\n')
                cif_info = pyNMBu.load_cif(self,cif_file,noOutput)
                    
                # Determine forms based on crystal type
                if self.crystal_type=='fcc' :
                    dist= pyNMBu.FindInterAtomicDist(self)# Extract the interatomic distance
                    index=0 
                    if not noOutput :
                        print(f" {bg.LIGHTGREENB} xyz/cif files can be created for {self.cif_name} of Bravais lattice={self.crystal_type}. \033[0m ")
   
                    # Create instances for each form and size
                    for i in self.sizes :
                        index += 1 
                        if not noOutput :
                            print(f'{bg.LIGHTBLUEB} Number of atoms per edge is {i+1}')
                        TestNP =cNP.fccdrDD(
                            element=element,
                            Rnn=dist,
                            nShell=i,
                            postAnalyzis=True,
                            aseView=False,
                            thresholdCoreSurface=1,
                            skipSymmetryAnalyzis=True,
                            noOutput= True
                        )

                        circumsphere_diameter=TestNP.radiusCircumscribedSphere*2*0.1 # Setting a maximal size of NPs : circumscribed sphere diameter
                        if circumsphere_diameter<max_size :
                            if not noOutput :
                                print(f'{bg.LIGHTBLUEB}Circumscribed sphere diameter ={circumsphere_diameter}\033[0m')
                            pyNMBu.writexyz_generalized_catalan(self,structure,path,TestNP,index,noOutput) 
                        else :
                            break

                
                elif self.crystal_type=='bcc' :
                    dist= pyNMBu.FindInterAtomicDist(self)# Extract the interatomic distance
                    index=0 
                    if not noOutput :
                        print(f" {bg.LIGHTGREENB} xyz/cif files can be created for {self.cif_name} of Bravais lattice={self.crystal_type}. \033[0m ")
   
                    # Create instances for each form and size
                    for i in self.sizes :
                        index += 1 
                        if not noOutput :
                            print(f'{bg.LIGHTBLUEB} Number of atoms per edge is {i+1}')
                        TestNP =cNP.bccrDD(
                            element=element,
                            Rnn=dist,
                            nShell=i,
                            postAnalyzis=True,
                            aseView=False,
                            thresholdCoreSurface=1,
                            skipSymmetryAnalyzis=True,
                            noOutput= True
                        )

                        circumsphere_diameter=TestNP.radiusCircumscribedSphere*2*0.1 # Setting a maximal size of NPs : circumscribed sphere diameter
                        if circumsphere_diameter<max_size :
                            if not noOutput :
                                print(f'{bg.LIGHTBLUEB}Circumscribed sphere diameter ={circumsphere_diameter}\033[0m')
                            pyNMBu.writexyz_generalized_catalan(self,structure,path,TestNP,index,noOutput) 
                        else :
                            break
                
                else : 
                    if not noOutput :
                        print(f" {bg.LIGHTREDB} xyz/cif files can't be created for {self.cif_name} because the crystal type isn't fcc.\033[0m  ")   
                    pass

          
            else :
                pass          

In [3]:
Catalan_Files?

Init signature:
Catalan_Files(
    path,
    cif_data,
    sizes,
    max_size: float = 50,
    noOutput: bool = True,
)
Docstring:     
A class for generating XYZ and CIF files of rhombic dodecahedron (bccrDD) 
and dihedral rhombic dodecahedron (fccdrDD) Catalan nanoparticles (NPs) 
from a dataset of compounds (CIF dataset). This process enables the creation 
of a well-structured database optimized for machine learning applications, 
ensuring consistency in format and data representation.
Init docstring:
Initialize the class with CIF data and sizes.
    Args:
path (str) : Path that will contain the created xyz files
cif data (DataFrame):  DataFrame containing CIF files of different compounds
sizes (array-like): Array of the number of shells, ex: [2,3,4]
max_size (float, optional) : Maximal size for the NPs, equals to the diameter of the circumscribed sphere, equals 50 nm by default.
noOutput (bool): If False, details of the files are printed.
    Methods:
create_catalan(noOutput, path

### Example of use of the  Catalan_class

##### Create all the possible forms from a dataset of CIF files

If noOutput=False :  
 - <span style="color:#008000"> in green : the allowed shapes created (fcc) </span> 
 - <span style="color:#FF0000">in red : the shapes that are not created (not fcc)</span>

In [4]:
t = pyNMBu.timer()
t.chrono_start()

# Path where the files are created
path='database_1_5nm/catalan_1_5nm'

# Sizes of the NPs
sizes=np.arange(1,30,1) # Control the sizes by the number of bonds per edges : here 1 to 25 bonds (step of 1)

# Possiblity to define a maximal size: max_size (a circumsphere diameter). If max_size isn't defined, the NPs will have the maximum number of bonds defined, here 25.
class_test=Catalan_Files(path,cif_data=data.pyNMBcif.CIFdf,sizes=sizes,max_size=5,noOutput= False) # Ex: the maximal size is a circumsphere diameter of 5 nm.



print(t.chrono_stop(hdelay=True))


                      Fe bcc                      

Absolute path to CIF: /home/sara/pyNanoMatBuilder/cif_database/cod5000217-Fe_bcc.cif
  xyz/cif files can be created for Fe bcc of Bravais lattice=bcc.  
 Number of atoms per edge is 2
Circumscribed sphere diameter =0.5721599999999999
  xyz file created:database_1_5nm/catalan_1_5nm/Fe_bcc_bccrDD_0000001_0000000.xyz 
  cif file created:database_1_5nm/catalan_1_5nm/Fe_bcc_bccrDD_0000001_0000000.cif
 Number of atoms per edge is 3
Circumscribed sphere diameter =1.1443199999999998
  xyz file created:database_1_5nm/catalan_1_5nm/Fe_bcc_bccrDD_0000002_0000000.xyz 
  cif file created:database_1_5nm/catalan_1_5nm/Fe_bcc_bccrDD_0000002_0000000.cif
 Number of atoms per edge is 4
Circumscribed sphere diameter =1.7164799999999998
  xyz file created:database_1_5nm/catalan_1_5nm/Fe_bcc_bccrDD_0000003_0000000.xyz 
  cif file created:database_1_5nm/catalan_1_5nm/Fe_bcc_bccrDD_0000003_0000000.cif
 Number of atoms per edge is 5
Circumscribed sphere diam

/home/sara/Python3/Debye_calc/lib/python3.11/site-packages/ase/io/cif.py:408: UserWarning: crystal system 'cubic' is not interpreted for space group Spacegroup(229, setting=1). This may result in wrong setting!
  warnings.warn(


Circumscribed sphere diameter =3.4329599999999996
  xyz file created:database_1_5nm/catalan_1_5nm/Fe_bcc_bccrDD_0000006_0000000.xyz 
  cif file created:database_1_5nm/catalan_1_5nm/Fe_bcc_bccrDD_0000006_0000000.cif
 Number of atoms per edge is 8
Circumscribed sphere diameter =4.013815359243335
  xyz file created:database_1_5nm/catalan_1_5nm/Fe_bcc_bccrDD_0000007_0000000.xyz 
  cif file created:database_1_5nm/catalan_1_5nm/Fe_bcc_bccrDD_0000007_0000000.cif
 Number of atoms per edge is 9
Circumscribed sphere diameter =4.591664491257335
  xyz file created:database_1_5nm/catalan_1_5nm/Fe_bcc_bccrDD_0000008_0000000.xyz 
  cif file created:database_1_5nm/catalan_1_5nm/Fe_bcc_bccrDD_0000008_0000000.cif
 Number of atoms per edge is 10

                     Mn alpha                     

Absolute path to CIF: /home/sara/pyNanoMatBuilder/cif_database/cod9011068-Mn_alpha.cif
  xyz/cif files can't be created for Mn alpha because the crystal type isn't fcc.  

                     Mn beta          